In [4]:
import os
import sys

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
spark = get_spark_session()
print("Project root:", PROJECT_ROOT)

from src.analytics.queries import ALL_QUERIES

Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform


### Read integrated taxi trip detlta table

In [5]:
from deltalake import DeltaTable
from pyspark.sql import functions as F

GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "gold", "integrated_taxi_trips")

def _read_delta(spark, path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"No data at {path} - run the integration pipeline first")
    files = DeltaTable(path).file_uris()
    return spark.read.option("basePath", path).parquet(*files)

gold = _read_delta(spark, GOLD_PATH).filter(F.col("is_valid"))
gold.createOrReplaceTempView("integrated_taxi_trips")
print("Rows in view:", spark.sql("SELECT COUNT(*) FROM integrated_taxi_trips").collect()[0][0])

Rows in view: 9538046


### Run queries

In [6]:
print(f"{len(ALL_QUERIES)} queries loaded:")
for name in ALL_QUERIES:
    print(" -", name)


report_lines = ["# Week 2 — Analytical Query Results", ""]
result_dfs = {}

for name, sql in ALL_QUERIES.items():
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    result_df = spark.sql(sql)
    display(result_df.limit(20).toPandas())

    result_dfs[name] = result_df
    report_lines.append(f"## {name}")
    report_lines.append("")
    pdf = result_df.limit(20).toPandas()
    report_lines.append(pdf.to_markdown(index=False))
    report_lines.append("")

6 queries loaded:
 - Q1_monthly_demand_per_zone
 - Q2_avg_distance_by_weather
 - Q3_air_quality_vs_demand
 - Q4_zones_most_weather_sensitive
 - Q5_peak_hour_per_day
 - Q6_monthly_trends

Q1_monthly_demand_per_zone


,pickup_zone,pickup_year,pickup_month,trip_count
0,JFK Airport,2024,1,144077
1,Upper East Side South,2024,1,141577
2,Midtown Center,2024,1,141355
3,Upper East Side North,2024,1,135679
4,Midtown East,2024,1,105492
5,Times Sq/Theatre District,2024,1,104750
6,Penn Station/Madison Sq West,2024,1,103833
7,Lincoln Square East,2024,1,102853
8,LaGuardia Airport,2024,1,88623
9,Upper West Side South,2024,1,87902



Q2_avg_distance_by_weather


,weather_condition,weather_condition_code,trip_count,avg_trip_distance_miles
0,Fair,2,3027082,4.281
1,Cloudy,3,2355419,3.948
2,Overcast,4,2066447,4.040
3,Light Rain,7,743432,3.811
4,Heavy Rain,9,402901,4.243
5,Rain,8,253824,3.719
6,Fog,5,235549,3.530
7,Clear,1,227418,3.800
8,Light Snowfall,14,143948,3.311
9,Snowfall,15,22219,3.038



Q3_air_quality_vs_demand


,pm25_level,trip_count,avg_pm25
0,1: Good (0.0-9.0),720334,4.81
1,2: Moderate (9.1-35.4),233234,15.27
2,3: Unhealthy for Sensitive Groups (35.5-55.4),1828,37.55


... (2 outputs left out)


Q5_peak_hour_per_day


,day_of_week,peak_hour,trip_count
0,Sunday,0,97535
1,Monday,22,79266
2,Tuesday,23,97242
3,Wednesday,23,108917
4,Thursday,23,119133
5,Friday,0,105205
6,Saturday,23,98696



Q6_monthly_trends


,pickup_year,pickup_month,trip_count,avg_fare,avg_distance
0,2024,1,2936541,18.19,3.66
1,2024,2,2995372,18.04,3.86
2,2024,3,3606133,18.66,4.50


### Save results

In [7]:
out_dir = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "analytics_results.md")

with open(out_path, "w") as f:
    f.write("\n".join(report_lines))

print(f"Saved: {out_path}")

Saved: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform/reports/analytics_results.md


In [8]:
from IPython.display import Markdown, display

display(Markdown("\n".join(report_lines)))

# Week 2 — Analytical Query Results

## Q1_monthly_demand_per_zone

| pickup_zone                  |   pickup_year |   pickup_month |   trip_count |
|:-----------------------------|--------------:|---------------:|-------------:|
| JFK Airport                  |          2024 |              1 |       144077 |
| Upper East Side South        |          2024 |              1 |       141577 |
| Midtown Center               |          2024 |              1 |       141355 |
| Upper East Side North        |          2024 |              1 |       135679 |
| Midtown East                 |          2024 |              1 |       105492 |
| Times Sq/Theatre District    |          2024 |              1 |       104750 |
| Penn Station/Madison Sq West |          2024 |              1 |       103833 |
| Lincoln Square East          |          2024 |              1 |       102853 |
| LaGuardia Airport            |          2024 |              1 |        88623 |
| Upper West Side South        |          2024 |              1 |        87902 |
| Midtown North                |          2024 |              1 |        84555 |
| Murray Hill                  |          2024 |              1 |        82243 |
| East Chelsea                 |          2024 |              1 |        76866 |
| Clinton East                 |          2024 |              1 |        75631 |
| Union Sq                     |          2024 |              1 |        75061 |
| Lenox Hill West              |          2024 |              1 |        73410 |
| Lenox Hill East              |          2024 |              1 |        65636 |
| West Village                 |          2024 |              1 |        65424 |
| East Village                 |          2024 |              1 |        65218 |
| Midtown South                |          2024 |              1 |        64105 |

## Q2_avg_distance_by_weather

| weather_condition   |   weather_condition_code |   trip_count |   avg_trip_distance_miles |
|:--------------------|-------------------------:|-------------:|--------------------------:|
| Fair                |                        2 |      3027082 |                     4.281 |
| Cloudy              |                        3 |      2355419 |                     3.948 |
| Overcast            |                        4 |      2066447 |                     4.04  |
| Light Rain          |                        7 |       743432 |                     3.811 |
| Heavy Rain          |                        9 |       402901 |                     4.243 |
| Rain                |                        8 |       253824 |                     3.719 |
| Fog                 |                        5 |       235549 |                     3.53  |
| Clear               |                        1 |       227418 |                     3.8   |
| Light Snowfall      |                       14 |       143948 |                     3.311 |
| Snowfall            |                       15 |        22219 |                     3.038 |
| Sleet               |                       12 |        17209 |                     3.248 |
| Heavy Sleet         |                       13 |        16724 |                     3.055 |
| Heavy Snowfall      |                       16 |        15848 |                     2.371 |
| Freezing Rain       |                       10 |        10026 |                     6.734 |

## Q3_air_quality_vs_demand

| pm25_level                                    |   trip_count |   avg_pm25 |
|:----------------------------------------------|-------------:|-----------:|
| 1: Good (0.0-9.0)                             |       720334 |       4.81 |
| 2: Moderate (9.1-35.4)                        |       233234 |      15.27 |
| 3: Unhealthy for Sensitive Groups (35.5-55.4) |         1828 |      37.55 |

## Q4_zones_most_weather_sensitive

| pickup_zone                  |   conditions_observed |   demand_stddev |   min_demand |   max_demand |
|:-----------------------------|----------------------:|----------------:|-------------:|-------------:|
| Midtown Center               |                    14 |         49159.7 |          412 |       140906 |
| Upper East Side South        |                    14 |         47734.1 |          676 |       138877 |
| JFK Airport                  |                    14 |         46056.8 |          443 |       140254 |
| Upper East Side North        |                    14 |         45518.8 |          650 |       135354 |
| Midtown East                 |                    14 |         36024   |          376 |       105039 |
| Times Sq/Theatre District    |                    14 |         34741   |          257 |       104078 |
| Penn Station/Madison Sq West |                    14 |         34147.1 |          369 |       103248 |
| Lincoln Square East          |                    14 |         33307.3 |          309 |       100595 |
| LaGuardia Airport            |                    14 |         31842   |          106 |        96203 |
| Upper West Side South        |                    14 |         30662.3 |          310 |        94012 |

## Q5_peak_hour_per_day

| day_of_week   |   peak_hour |   trip_count |
|:--------------|------------:|-------------:|
| Sunday        |           0 |        97535 |
| Monday        |          22 |        79266 |
| Tuesday       |          23 |        97242 |
| Wednesday     |          23 |       108917 |
| Thursday      |          23 |       119133 |
| Friday        |           0 |       105205 |
| Saturday      |          23 |        98696 |

## Q6_monthly_trends

|   pickup_year |   pickup_month |   trip_count |   avg_fare |   avg_distance |
|--------------:|---------------:|-------------:|-----------:|---------------:|
|          2024 |              1 |  2.93654e+06 |      18.19 |           3.66 |
|          2024 |              2 |  2.99537e+06 |      18.04 |           3.86 |
|          2024 |              3 |  3.60613e+06 |      18.66 |           4.5  |
